# 📡 NewsBreakout - RSS Feed News Crawler & Tester

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook is specifically designed to test fetching, parsing, and normalizing news articles by **simply pasting any RSS Feed URL** (CNN, BBC, Reuters, NYT, VnExpress, The Guardian, etc.) following the official **Article Schema** of the **NewsBreakout** project.

---

### 🎯 Pipeline Overview:
1. **Paste RSS Feed URL**: Parse the feed structure and retrieve the latest article entries in real-time.
2. **Full-Text Article Extraction**: Use `trafilatura` to extract clean article body text while stripping boilerplate, ads, and navigation elements.
3. **Automatic Metadata Detection**: Identify publisher name (`publisher`), country code (`country`), and language (`language`).
4. **NewsBreakout Schema Compliance**: Normalize timestamps to ISO-8601 UTC and generate a unique SHA-256 `article_id`.
5. **Data Quality & Export**: Inspect missing values, calculate word counts, preview markdown articles, and export to `.jsonl` and `.csv`.

## 🛠️ 1. Install Dependencies

In [ ]:
!pip install -q feedparser trafilatura beautifulsoup4 lxml pandas tqdm tldextract python-dateutil requests

## 📦 2. Initialize RSS Crawler & Content Extractor Engine

In [ ]:
import os
import re
import json
import time
import hashlib
import datetime
import requests
import feedparser
import trafilatura
import tldextract
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from dateutil import parser as date_parser
from tqdm import tqdm
from IPython.display import display, Markdown

# Standard User-Agent header to avoid bot-blocking
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,vi;q=0.8"
}

# Known publisher domain mapping
KNOWN_PUBLISHERS = {
    "cnn.com": ("CNN", "US"),
    "bbc.com": ("BBC", "GB"),
    "bbc.co.uk": ("BBC", "GB"),
    "nytimes.com": ("The New York Times", "US"),
    "reuters.com": ("Reuters", "US"),
    "theguardian.com": ("The Guardian", "GB"),
    "washingtonpost.com": ("The Washington Post", "US"),
    "bloomberg.com": ("Bloomberg", "US"),
    "techcrunch.com": ("TechCrunch", "US"),
    "theverge.com": ("The Verge", "US"),
    "vnexpress.net": ("VnExpress", "VN"),
    "tuoitre.vn": ("Tuổi Trẻ", "VN"),
    "thanhnien.vn": ("Thanh Niên", "VN"),
    "vietnamnet.vn": ("VietNamNet", "VN"),
    "dantri.com.vn": ("Dân Trí", "VN")
}

def generate_article_id(url: str, title: str = "") -> str:
    """Generate a unique 16-character SHA-256 ID based on canonical URL or title."""
    clean_u = url.split("?")[0].strip()
    key = clean_u if clean_u else (title or "").strip()
    return hashlib.sha256(key.encode('utf-8')).hexdigest()[:16]

def parse_iso_date(date_str: str) -> str:
    """Parse and standardize publication date to ISO 8601 UTC format."""
    if not date_str:
        return datetime.datetime.now(datetime.timezone.utc).isoformat()
    try:
        dt = date_parser.parse(str(date_str))
        if not dt.tzinfo:
            dt = dt.replace(tzinfo=datetime.timezone.utc)
        return dt.isoformat()
    except Exception:
        return datetime.datetime.now(datetime.timezone.utc).isoformat()

def detect_publisher_and_country(url: str, feed_title: str = ""):
    """Detect Publisher Name and Country code from domain or feed title."""
    ext = tldextract.extract(url)
    domain_key = f"{ext.domain}.{ext.suffix}".lower()
    
    if domain_key in KNOWN_PUBLISHERS:
        return KNOWN_PUBLISHERS[domain_key]
    
    publisher = ext.domain.capitalize() if ext.domain else (feed_title or "Unknown")
    country = ext.suffix.upper() if len(ext.suffix) == 2 else "US"
    return publisher, country

def extract_article_body_and_meta(url: str) -> dict:
    """
    Fetch article URL and extract full body text and metadata using trafilatura.
    """
    clean_url = url.split("?")[0].strip()
    html_content = None
    
    try:
        html_content = trafilatura.fetch_url(clean_url, headers=DEFAULT_HEADERS)
    except Exception:
        pass
        
    if not html_content:
        try:
            resp = requests.get(clean_url, headers=DEFAULT_HEADERS, timeout=12)
            if resp.status_code == 200:
                html_content = resp.text
        except Exception:
            return None
            
    if not html_content:
        return None

    # Extract metadata & clean body text via trafilatura
    meta_json = trafilatura.extract(
        html_content,
        output_format="json",
        include_comments=False,
        include_tables=False,
        with_metadata=True
    )
    
    data = json.loads(meta_json) if meta_json else {}
    body = data.get("text") or ""
    
    # Fallback to BeautifulSoup if trafilatura fails to extract body
    if not body:
        soup = BeautifulSoup(html_content, "html.parser")
        p_tags = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 40]
        body = "\n\n".join(p_tags)
        
    return {
        "body": body,
        "authors": data.get("author") or "",
        "extracted_date": data.get("date") or "",
        "extracted_desc": data.get("description") or "",
        "language": data.get("language") or "en",
        "extracted_category": data.get("categories") or ""
    }

print("✅ RSS Crawler Engine initialized and ready!")

## 🎯 3. Paste RSS Feed URL for Testing

> 💡 **Example RSS Feed URLs to test**:
> - **CNN (Top Stories)**: `http://rss.cnn.com/rss/cnn_topstories.rss`
> - **CNN (World)**: `http://rss.cnn.com/rss/cnn_world.rss`
> - **CNN (Technology)**: `http://rss.cnn.com/rss/cnn_tech.rss`
> - **BBC News (World)**: `https://feeds.bbci.co.uk/news/world/rss.xml`
> - **VnExpress (World)**: `https://vnexpress.net/rss/the-gioi.rss`
> - **VnExpress (Latest)**: `https://vnexpress.net/rss/tin-moi-nhat.rss`
> - **Tuổi Trẻ (Current Affairs)**: `https://tuoitre.vn/rss/thoi-su.rss`

In [ ]:
# @title 📡 RSS Feed Crawler Configuration Form

# @markdown Paste RSS Feed URL to test:
rss_feed_url = "http://rss.cnn.com/rss/cnn_topstories.rss" # @param {type:"string"}

# @markdown Number of articles to crawl:
max_articles = 5 # @param {type:"slider", min:1, max:30, step:1}

# @markdown Custom category (leave empty to auto-detect from RSS/URL):
custom_category = "" # @param {type:"string"}

# ==================================================
# EXECUTE RSS FEED COLLECTION
# ==================================================
clean_rss_url = rss_feed_url.strip()
articles_data = []

if not clean_rss_url:
    print("❌ Please enter an RSS Feed URL in 'rss_feed_url'!")
else:
    print(f"📡 Fetching and parsing RSS Feed: {clean_rss_url}")
    feed = feedparser.parse(clean_rss_url)
    
    feed_title = getattr(feed.feed, "title", "Unknown Feed")
    feed_desc = getattr(feed.feed, "description", "")
    total_entries = len(feed.entries)
    
    print(f"✅ Successfully loaded RSS Feed: [{feed_title}]")
    print(f"📌 Total entries in feed: {total_entries}")
    
    selected_entries = feed.entries[:max_articles]
    print(f"🚀 Scraping latest {len(selected_entries)} articles...\n")
    
    for entry in tqdm(selected_entries, desc="Scraping RSS Articles"):
        raw_url = getattr(entry, "link", "")
        clean_url = raw_url.split("?")[0].strip()
        
        if not clean_url:
            continue
            
        title = getattr(entry, "title", "").strip()
        
        # Extract summary from RSS feed
        feed_summary = getattr(entry, "summary", "")
        if feed_summary:
            feed_summary = BeautifulSoup(feed_summary, "html.parser").get_text().strip()
            
        published_raw = getattr(entry, "published", "")
        rss_author = getattr(entry, "author", "")
        
        # Detect Publisher and Country
        publisher, country = detect_publisher_and_country(clean_url, feed_title)
        
        # Extract full article body & metadata
        article_details = extract_article_body_and_meta(clean_url)
        time.sleep(0.4) # Polite delay between requests
        
        body = ""
        authors = rss_author
        final_desc = feed_summary
        final_pub_date = published_raw
        language = "en"
        cat = custom_category.strip()
        
        if article_details:
            body = article_details.get("body", "")
            if not authors:
                authors = article_details.get("authors", "")
            if not final_desc:
                final_desc = article_details.get("extracted_desc", "")
            if not final_pub_date:
                final_pub_date = article_details.get("extracted_date", "")
            language = article_details.get("language", "en")
            if not cat:
                extracted_cat = article_details.get("extracted_category", "")
                if isinstance(extracted_cat, list):
                    cat = ", ".join(extracted_cat)
                else:
                    cat = str(extracted_cat)
                    
        # Fallback category determination from tags or URL path
        if not cat:
            if hasattr(entry, "tags") and entry.tags:
                cat = entry.tags[0].get("term", "")
            else:
                path_parts = [p for p in urlparse(clean_url).path.split("/") if p and not p.endswith((".html", ".htm"))]
                cat = path_parts[0].capitalize() if path_parts and not path_parts[0].isdigit() else "General"

        # Create normalized record matching NewsBreakout Article Schema
        record = {
            "article_id": generate_article_id(clean_url, title),
            "title": title,
            "description": final_desc,
            "body": body,
            "publisher": publisher,
            "url": clean_url,
            "published_time": parse_iso_date(final_pub_date),
            "country": country,
            "language": language,
            "category": cat,
            "authors": authors
        }
        articles_data.append(record)

    df_results = pd.DataFrame(articles_data)
    print(f"\n🎉 RSS Scraping completed! Successfully extracted {len(df_results)} articles.")

## 📊 4. Data Quality & Schema Verification

In [ ]:
if not df_results.empty:
    # Calculate body word count
    df_results["word_count"] = df_results["body"].apply(lambda x: len(str(x).split()) if str(x) else 0)
    
    print("📋 COLLECTED ARTICLES OVERVIEW:")
    display(df_results[["article_id", "publisher", "title", "category", "published_time", "word_count"]])
    
    print("\n🔍 SCHEMA COMPLETENESS CHECK:")
    schema_fields = ["article_id", "title", "description", "body", "publisher", "url", "published_time", "country", "language", "category", "authors"]
    for field in schema_fields:
        empty_count = sum(df_results[field].apply(lambda x: 1 if not str(x).strip() else 0))
        status = "✅ Complete (100%)" if empty_count == 0 else f"⚠️ Empty: {empty_count}/{len(df_results)}"
        print(f" - {field:15s}: {status}")
        
    print("\n📈 BODY WORD COUNT DISTRIBUTION:")
    print(df_results["word_count"].describe())
else:
    print("No article data available.")

## 🔎 5. Sample Article Preview

In [ ]:
if not df_results.empty:
    sample = df_results.iloc[0]
    
    preview_md = f"""
### 📰 [{sample['publisher']}] {sample['title']}
- **Article ID**: `{sample['article_id']}`
- **Authors**: `{sample['authors'] or 'N/A'}`
- **Published Time**: `{sample['published_time']}`
- **Category**: `{sample['category']}` | **Language**: `{sample['language']}` | **Country**: `{sample['country']}`
- **Body Word Count**: `{sample['word_count']} words`
- **Canonical URL**: [{sample['url']}]({sample['url']})

---
#### 📝 Description / Summary:
> {sample['description'] or 'No description provided'}

#### 📄 Full Body Preview (first 700 characters):
```text
{sample['body'][:700]}...
```
"""
    display(Markdown(preview_md))
else:
    print("No article to display.")

## 💾 6. Export Dataset (JSON Lines & CSV)

In [ ]:
if not df_results.empty:
    out_df = df_results.drop(columns=["word_count"], errors="ignore")
    
    jsonl_filename = "rss_articles_sample.jsonl"
    csv_filename = "rss_articles_sample.csv"
    
    # Export to JSON Lines (standard format for NLP / embeddings pipeline)
    out_df.to_json(jsonl_filename, orient="records", lines=True, force_ascii=False)
    # Export to CSV with UTF-8 BOM encoding for seamless Excel compatibility
    out_df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    
    print(f"💾 Successfully exported {len(out_df)} articles:")
    print(f"   1. {jsonl_filename} (JSONL format)")
    print(f"   2. {csv_filename} (CSV format)")
    
    # Optional: trigger direct browser download when running in Google Colab
    try:
        from google.colab import files
        print("\n📥 Download files to your local machine in Colab: files.download('rss_articles_sample.jsonl')")
    except ImportError:
        pass